Setting the python path to import code from app/src.

The extrapath is configured at .vscode/settings.json to avoid IDE errors.

In [ ]:
import sys
from pathlib import Path


def find_project_root(marker="pyproject.toml") -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"{marker} not found starting from {p}")


root_dir = find_project_root() / "app" / "src"
sys.path.append(str(root_dir))

Importing the necessary modules and classes for the classification task.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI

from domain.clause_tree import Clause, ClauseTree, ClauseTreeReport
from infrastructure.parsing.rules_loader import load_classification_rules
from infrastructure.parsing.llm_classifier import LangchainClauseClassifier
from application.use_cases.clause_classification import classify_and_enrich_clauses

load_dotenv()


In this case, I'll use OpenRouter as the LLM provider through the LangChain library designed for OpenAI API-compatible LLMs.

In [ ]:
llm = ChatOpenAI(
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
    model=os.getenv("LLM_MODEL_FAST"), 
    temperature=0.0,
)


classifier = LangchainClauseClassifier(llm=llm)

Creating some mock data.

In [ ]:
c1 = Clause(
    document_id="1", clause_id="c1", numbering_label="1.",
    title="Riscos Excluídos", convention="numbered_decimal",
    depth=1, parent_id=None, child_ids=(),
    content_lines=("Não haverá cobertura para os seguintes eventos...",),
    page_start=1, page_end=1
)

c2 = Clause(
    document_id="1", clause_id="c2", numbering_label="2.",
    title="Limites Máximos de Indenização", convention="numbered_decimal",
    depth=1, parent_id=None, child_ids=(),
    content_lines=("O limite máximo de indenização será o valor de mercado referenciado na data do sinistro...",),
    page_start=1, page_end=1
)

tree = ClauseTree(
    document_id="1",
    filename="15414610650202459.pdf",
    roots=(c1, c2),
    all_clauses=(c1, c2),
    report=ClauseTreeReport("1", "15414610650202459.pdf", 2, 1, 0, 100, 0.0, "text", ())
)


manifest_records = [
    {
        "id": "1",
        "susep_process": "15414.610650/2024-59",
        "insurer": "PORTO SEGURO COMPANHIA DE SEGUROS GERAIS",
        "cnpj": "61198164000160",
        "product_line": "CASCO",
        "indemnity_regime": "VD",
        "process_year": "2024"
    }
]


In [ ]:
csv_path = Path("../../data/parsing/clause_type_mapping.csv")
rules = load_classification_rules(csv_path)

print(f"Total of rules: {len(rules)}")
print(rules)
for rule in rules:
    print(rule)


Running the classification and enrichment of clauses.

In [ ]:
enriched_clauses = classify_and_enrich_clauses(
    tree=tree,
    manifest_records=manifest_records,
    rules=rules,
    classifier=classifier
)


for tc in enriched_clauses:
    print("\n\n")
    print(f"Original title: {tc.clause.title}")
    print(f"inferred type: {tc.clause_type.name}")
    print(f"Source: {tc.type_source.name}")
    print(f"Confidence: {tc.confidence}")
    print(f"Provenance: {tc.provenance.insurer} (CNPJ: {tc.provenance.cnpj})")
    print(f"SUSEP's Process: {tc.provenance.susep_process}")
